In [1]:
# 1. Install system packages
!apt-get update -qq
!apt-get install -y -qq perl   libxml2-dev   libxml-libxml-perl
# the XML::LibXML Perl module

# 2. (Optional) If libxml-libxml-perl wasn’t available via apt,
#    install via cpanminus instead:
!apt-get install -y -qq cpanminus
!cpanm --notest XML::LibXML


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
(Reading database ... 126332 files and directories currently installed.)
Preparing to unpack .../libperl5.34_5.34.0-3ubuntu1.4_amd64.deb ...
Unpacking libperl5.34:amd64 (5.34.0-3ubuntu1.4) over (5.34.0-3ubuntu1.3) ...
Preparing to unpack .../perl_5.34.0-3ubuntu1.4_amd64.deb ...
Unpacking perl (5.34.0-3ubuntu1.4) over (5.34.0-3ubuntu1.3) ...
Preparing to unpack .../perl-base_5.34.0-3ubuntu1.4_amd64.deb ...
Unpacking perl-base (5.34.0-3ubuntu1.4) over (5.34.0-3ubuntu1.3) ...
Setting up perl-base (5.34.0-3ubuntu1.4) ...
(Reading database ... 126332 files and directories currently installed.)
Preparing to unpack .../00-perl-modules-5.34_5.34.0-3ubuntu1.4_all.deb ...
Unpacking perl-modules-5.34 (5.34.0-3ubuntu1.4) over (5.34.0-3ubuntu1.3) ...
Selecting previously u

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## SRD Functions

In [3]:
import os
import pickle as pkl
import numpy
import sys

In [4]:
def norm_id_inkml(root_path, out_path, paths):
    process_num = 0
    for path in paths:
        file_list  = os.listdir(root_path + path)
        for file_name in file_list:
            inkml_file = root_path + path + '/' + file_name
            norm_inkml_path = out_path + path
            if os.path.exists(norm_inkml_path):
                norm_inkml_file = norm_inkml_path + '/' + file_name
            else:
                os.mkdir(norm_inkml_path)
                norm_inkml_file = norm_inkml_path + '/' + file_name
            #print inkml_file
            with open(inkml_file, encoding='utf-8', errors='replace') as f:
                lines = f.readlines()
                norm_lines = []
                sub_dict = {}
                symbol_stack = []
                mroot_line_list = []
                i = 0
                line_tmp = lines[i].strip()
                while line_tmp != '</annotationXML>': #or i < len(lines):
                    if line_tmp.find('supsub') != -1:
                        print('warning: this inkml has supsub structure', inkml_file)
                        sys.exit()
                    if line_tmp.find('xml:id') == -1:
                        norm_lines.append(lines[i])
                        i += 1
                        if i >= len(lines):
                            break
                        line_tmp = lines[i].strip()
                    else:
                        if line_tmp[:6] == '<mfrac':
                            symbol_stack.append('frac')
                            idx = symbol_stack.count('frac')
                            sub_element = line_tmp.split('"')[1]
                            sub_source_str = '"' + sub_element + '"'
                            sub_target_str = '"' + 'frac_' + str(idx) + '"'
                            sub_dict[sub_source_str] = sub_target_str
                            norm_lines.append(lines[i].replace(sub_source_str, sub_target_str))
                        elif line_tmp[:6] == '<msqrt':
                            symbol_stack.append('sqrt')
                            idx = symbol_stack.count('sqrt')
                            sub_element = line_tmp.split('"')[1]
                            sub_source_str = '"' + sub_element + '"'
                            sub_target_str = '"' + 'sqrt_' + str(idx) + '"'
                            sub_dict[sub_source_str] = sub_target_str
                            norm_lines.append(lines[i].replace(sub_source_str, sub_target_str))
                        elif line_tmp[:6] == '<mroot':
                            print('this inkml has root structure', inkml_file)
                            symbol_stack.append('sqrt')
                            idx = symbol_stack.count('sqrt')
                            sub_element = line_tmp.split('"')[1]
                            sub_source_str = '"' + sub_element + '"'
                            sub_target_str = '"' + 'sqrt_' + str(idx) + '"'
                            sub_dict[sub_source_str] = sub_target_str
                            norm_lines.append(lines[i].replace(sub_source_str, sub_target_str))
                            j = i + 1
                            while j < len(lines):
                                root_line_tmp = lines[j].strip()
                                if root_line_tmp != '</mroot>':
                                    j += 1
                                    if root_line_tmp[:6] == '<mroot':
                                        print('this inkml has loop root structure', inkml_file)
                                        break
                                        # sys.exit()
                                else:
                                    mroot_idx = j-1
                                    mroot_line_tmp = lines[mroot_idx].strip()
                                    if mroot_line_tmp[:3] != '<mn' and mroot_line_tmp[:3] != '<mi':
                                        print('this inkml has special root structure', inkml_file)
                                        break
                                    else:
                                        mroot_line_list.append(mroot_idx)
                                        symbol_beg = mroot_line_tmp.find('>') + 1
                                        symbol_end = mroot_line_tmp.rfind('<')
                                        if symbol_end < symbol_beg:
                                            print('this line', mroot_line_tmp, '>*< wrong')
                                            i += 1
                                            # if i > len(lines):
                                            #     break
                                            line_tmp = lines[i].strip()
                                            continue
                                            # sys.exit()
                                        symbol = mroot_line_tmp[symbol_beg:symbol_end].strip()
                                        if symbol[0] == '\\':
                                            print('warning: this inkml has complex root structure', inkml_file)
                                            symbol = symbol[1:]
                                        symbol_stack.append(symbol)
                                        idx = symbol_stack.count(symbol)
                                        sub_element = mroot_line_tmp.split('"')[1]
                                        sub_source_str = '"' + sub_element + '"'
                                        sub_target_str = '"' + symbol + '_' + str(idx) + '"'
                                        sub_dict[sub_source_str] = sub_target_str
                                        lines[mroot_idx] = lines[mroot_idx].replace(sub_source_str, sub_target_str)
                                        break
                            if j == len(lines):
                                print('this inkml has wrong root structure', inkml_file)
                                sys.exit()
                        else:
                            if i in mroot_line_list:
                                norm_lines.append(lines[i])
                            else:
                                symbol_beg = line_tmp.find('>') + 1
                                symbol_end = line_tmp.rfind('<')
                                if symbol_end < symbol_beg:
                                    print('this line', line_tmp, '>*< wrong')
                                    i += 1
                                    # if i > len(lines):
                                    #     break
                                    line_tmp = lines[i].strip()
                                    continue
                                    # sys.exit()
                                symbol = line_tmp[symbol_beg:symbol_end].strip()
                                if symbol[0] == '\\':
                                    symbol = symbol[1:]
                                symbol_stack.append(symbol)
                                idx = symbol_stack.count(symbol)
                                sub_element = line_tmp.split('"')[1]
                                sub_source_str = '"' + sub_element + '"'
                                sub_target_str = '"' + symbol + '_' + str(idx) + '"'
                                sub_dict[sub_source_str] = sub_target_str
                                norm_lines.append(lines[i].replace(sub_source_str, sub_target_str))
                        i += 1
                        line_tmp = lines[i].strip()
                while i < len(lines):
                    line_tmp = lines[i].strip()
                    if line_tmp[:19] != '<annotationXML href':
                        norm_lines.append(lines[i])
                    else:
                        line_sub_tmp = lines[i]
                        for sub_str in sub_dict:
                            if line_sub_tmp.find(sub_str) != -1:
                                line_sub_tmp = line_sub_tmp.replace(sub_str, sub_dict[sub_str])
                                break
                        norm_lines.append(line_sub_tmp)
                    i += 1

            if len(lines) != len(norm_lines):
                print('this inkml', inkml_file, 'processing error')
                sys.exit()
            f_out = open(norm_inkml_file,'w')
            for norm_line in norm_lines:
                f_out.write(norm_line)

            process_num += 1
            if process_num / 1000 == process_num * 1.0 / 1000:
                print('process files', process_num)



def inkml2lg(inkml_path, out_path, paths, crohmelib_bin_path):
    process_num = 0
    for path in paths:
        file_list  = os.listdir(inkml_path + path + '/CROHME2023_' + path)
        for file_name in file_list:
            inkml_file = inkml_path + path + '/CROHME2023_' + path + file_name
            print(inkml_file)
            lg_path = out_path + path
            if os.path.exists(lg_path):
                lg_file = lg_path + '/' + file_name[:-6] + '.lg'
            else:
                os.mkdir(lg_path)
                lg_file = lg_path + '/' + file_name[:-6] + '.lg'
            print(lg_file)

            order = 'perl ' + crohmelib_bin_path + 'crohme2lg.pl -s ' + inkml_file + ' ' + lg_file
            try:
                os.system(order)
            except:
                print('this file', inkml_file, 'inkml2lg pl error')
                # sys.exit()
            else:
                process_num += 1

            if process_num / 1000 == process_num * 1.0 / 1000:
                print('process files', process_num)

def norm_lg_v2(root_path, out_path, paths):
    process_num = 0
    for path in paths:
        file_list  = os.listdir(root_path + path)
        for file_name in file_list:
            lg_file = root_path + path + '/' + file_name
            norm_lg_path = out_path + path
            if os.path.exists(norm_lg_path):
                norm_lg_file = norm_lg_path + '/' + file_name
            else:
                os.mkdir(norm_lg_path)
                norm_lg_file = norm_lg_path + '/' + file_name
            with open(lg_file) as f:
                lines = f.readlines()
                norm_lines = []
                sub_dict = {}
                AUTO_num = 0
                for line in lines:
                    parts = line.strip().split(', ')
                    if parts[0] == 'O':
                        sym_pos = parts[1]
                        sym_pos_tmp = sym_pos.split('_')
                        sym = parts[2]
                        if sym_pos_tmp[0] == 'AUTO':
                            AUTO_num += 1
                        elif sym == '-' and sym_pos_tmp[0] == 'frac':
                            parts[2] = '\\frac'
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '-' and sym_pos_tmp[0] == '=':
                            parts[2] = '='
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '-' and sym_pos_tmp[0] == '=':
                            parts[2] = '='
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\lt':
                            parts[2] = '<'
                            sym_pos_tmp[0] = '<'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            if sym_pos != sym_pos_sub:
                                for test_line in lines:
                                    if test_line.find(sym_pos_sub) != -1:
                                        print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                        sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\gt':
                            parts[2] = '>'
                            sym_pos_tmp[0] = '>'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            if sym_pos != sym_pos_sub:
                                for test_line in lines:
                                    if test_line.find(sym_pos_sub) != -1:
                                        print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                        sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\ldots' and sym_pos_tmp[0] == 'ctdot':
                            sym_pos_tmp[0] = 'ldots'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\neq' and sym_pos_tmp[0] == 'ne':
                            sym_pos_tmp[0] = 'neq'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\lim' and sym_pos_tmp[0] == 'im':
                            sym_pos_tmp[0] = 'lim'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\infty' and sym_pos_tmp[0] == 'infin':
                            sym_pos_tmp[0] = 'infty'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\rightarrow' and sym_pos_tmp[0] == 'rarr':
                            sym_pos_tmp[0] = 'rightarrow'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\leq' and sym_pos_tmp[0] == 'le':
                            sym_pos_tmp[0] = 'leq'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\geq' and sym_pos_tmp[0] == 'ge':
                            sym_pos_tmp[0] = 'geq'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\exists' and sym_pos_tmp[0] == 'exist':
                            sym_pos_tmp[0] = 'exists'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        elif sym == '\\ldots' and sym_pos_tmp[0] == 'hellip':
                            sym_pos_tmp[0] = 'ldots'
                            sym_pos_sub = '_'.join(sym_pos_tmp)
                            for test_line in lines:
                                if test_line.find(sym_pos_sub) != -1:
                                    print('lg file norm error', lg_file, 'sub occupied', sym_pos_sub)
                                    sys.exit()
                            sub_dict[sym_pos] = sym_pos_sub
                            parts[1] = sym_pos_sub
                            norm_lines.append(', '.join(parts) + '\n')
                        else:
                            norm_lines.append(line)
                    elif parts[0] == 'EO':
                    # elif parts[0] == 'R':
                        if len(sub_dict) > 0:
                            line_sub_tmp = line
                            for sub_str in sub_dict:
                                # if line_sub_tmp.find(sub_str) != -1:
                                line_sub_tmp = line_sub_tmp.replace(sub_str, sub_dict[sub_str])
                                    # break
                            norm_lines.append(line_sub_tmp)
                        else:
                            norm_lines.append(line)
                    else:
                        norm_lines.append(line)

            if len(lines) - AUTO_num != len(norm_lines):
                print('this lg', lg_file, 'processing error')
                sys.exit()

            f_out = open(norm_lg_file,'w')
            for norm_line in norm_lines:
                f_out.write(norm_line)
            f_out.close()
            #sys.exit()

            process_num += 1
            if process_num / 1000 == process_num * 1.0 / 1000:
                print('process files', process_num)

def get_srd_label(root_path, out_path, paths, latex_caption_file):
    # RIT_2014_19 need manually revised
    # latex_caption_file = '/lustre1/hw/jszhang6/HMER/srd/data/caption/train_data_v1.txt'
    latex_caption = {}
    with open(latex_caption_file) as f:
        lines = f.readlines()
        for line in lines:
            parts = line.strip().split('\t')
            symbol_stack = []
            if len(parts) == 2:
                key = parts[0]
                caption = parts[1]
                caption = caption.replace(',', 'COMMA')
                # caption = caption.replace('{', '')
                # caption = caption.replace('}', '') # will destroy \{ \}
                caption = caption.replace('\\limits', '')
                caption = caption.replace('\cdots', '\ldots') # ctdot question
                caption = caption.replace('\cdot', '.') # ctdot question
                caption = caption.strip().split()
                id_caption = []
                for sym in caption:
                    if sym == '{' or sym == '}':
                        continue
                    elif sym == '_' or sym == '^':
                        id_caption.append(sym)
                    elif sym == 'COMMA':
                        symbol_stack.append('COMMA')
                        idx = symbol_stack.count('COMMA')
                        sub_symbol = 'COMMA_' + str(idx)
                        id_caption.append(sub_symbol)
                    elif len(sym) > 1:
                        if sym[0] != '\\':
                            print('this file', key, 'caption', parts[1], 'has informal symbol', sym)
                            sys.exit()
                        else:
                            sym = sym[1:] # remove \
                            symbol_stack.append(sym)
                            idx = symbol_stack.count(sym)
                            sub_symbol = sym + '_' + str(idx)
                            id_caption.append(sub_symbol)
                    elif len(sym) == 1:
                        symbol_stack.append(sym)
                        idx = symbol_stack.count(sym)
                        sub_symbol = sym + '_' + str(idx)
                        id_caption.append(sub_symbol)
                    else:
                        print('this file', key, 'caption', parts[1], 'has unknown symbol', sym)
                        sys.exit()

                latex_caption[key] = id_caption

    process_num = 0
    for path in paths:
        file_list  = os.listdir(root_path + path)
        for file_name in file_list:
            lg_file = root_path + path + '/' + file_name
            key = file_name[:-3] # remove .lg
            srd_path = out_path + path
            if os.path.exists(srd_path):
                srd_file = srd_path + '/' + key + '.srd'
            else:
                os.mkdir(srd_path)
                srd_file = srd_path + '/' + key + '.srd'

            sym_segment = {}
            sym_relation = {}
            with open(lg_file) as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split(', ', 4)
                    if parts[0] == 'O':
                        if parts[3] != '1.0':
                            print('this lg file Objects is informal', lg_file)
                            sys.exit()
                        if parts[1] not in sym_segment:
                            sym_segment[parts[1]] = parts[2] + '\t' + parts[4]
                        else:
                            print('this lg file has occupied sym_pos', lg_file)
                            sys.exit()

                    if parts[0] == 'EO':
                    # if parts[0] == 'R':
                        if parts[4] != '1.0':
                            print('this lg file SRT is informal', lg_file)
                            sys.exit()
                        if parts[2] not in sym_relation:
                            sym_relation[parts[2]] = []
                            sym_relation[parts[2]].append([parts[1],parts[3]])
                        else:
                            sym_relation[parts[2]].append([parts[1],parts[3]])
            # print key
            f_out = open(srd_file,'w')
            id_caption = latex_caption[key]
            if path == 'MathBrush':
                if 'ldots_1' in id_caption and 'ldots_1' not in sym_segment and '._1' in sym_segment:
                    new_symbol_stack = []
                    new_caption = []
                    new_id_caption = []
                    for id_cap in id_caption:
                        if id_cap == '_' or id_cap == '^':
                            new_caption.append(id_cap)
                        else:
                            new_caption.append(id_cap.split('_')[0])
                    new_caption_str = ' '.join(new_caption)
                    new_caption_str = new_caption_str.replace('ldots','. . .')
                    new_caption = new_caption_str.split(' ')
                    for sym in new_caption:
                        if sym == '_' or sym == '^':
                            new_id_caption.append(sym)
                        else:
                            new_symbol_stack.append(sym)
                            idx = new_symbol_stack.count(sym)
                            sub_symbol = sym + '_' + str(idx)
                            new_id_caption.append(sub_symbol)

                    id_caption = new_id_caption

            out_str = sym_segment[id_caption[0]] + '\t<s>\t-1\tStart' + '\n'
            f_out.write(out_str)
            if len(id_caption) > 1:
                for sym in id_caption[1:]:
                    if sym == '_' or sym == '^':
                        continue
                    elif sym not in sym_segment:
                        if sym.split('_')[0] == '[' or sym.split('_')[0] == ']':
                            continue
                        print('this lg file has unknown objects', lg_file)
                        print(' '.join(id_caption))
                        print(sym)
                        sys.exit()
                    elif sym not in sym_relation:
                        if sym.split('_')[0] == '[' or sym.split('_')[0] == ']':
                            continue
                        print('this lg file has unknown SRT', lg_file)
                        print(' '.join(id_caption))
                        print(sym)
                        sys.exit()
                    else:
                        out_str = sym_segment[sym]
                        if len(sym_relation[sym]) == 1:
                            out_str += '\t' + sym_segment[sym_relation[sym][0][0]] + '\t' + sym_relation[sym][0][1]
                            if sym_relation[sym][0][1] == 'Above':
                                previous_sym = id_caption[id_caption.index(sym) - 1]
                                relation_sym = sym_relation[sym][0][0].split('_')[0]
                                if previous_sym != '^' and relation_sym != 'frac' and relation_sym != 'sqrt':
                                    print('sup relation is wrong', lg_file)
                                    sys.exit()
                            if sym_relation[sym][0][1] == 'Below':
                                previous_sym = id_caption[id_caption.index(sym) - 1]
                                relation_sym = sym_relation[sym][0][0].split('_')[0]
                                if previous_sym != '_' and relation_sym != 'frac':
                                    print('sub relation is wrong', lg_file)
                                    sys.exit()
                        else:
                            # print 'this lg file has multiple node relation', lg_file
                            distance_list = []
                            for i in range(len(sym_relation[sym])):
                                distance = id_caption.index(sym) - id_caption.index(sym_relation[sym][i][0])
                                if distance <= 0:
                                    print('relation is not forward', lg_file)
                                    sys.exit()
                                else:
                                    distance_list.append(distance)
                            min_index = distance_list.index(min(distance_list))
                            out_str += '\t' + sym_segment[sym_relation[sym][min_index][0]] + '\t' + sym_relation[sym][min_index][1]
                            if sym_relation[sym][min_index][1] == 'Above':
                                previous_sym = id_caption[id_caption.index(sym) - 1]
                                relation_sym = sym_relation[sym][min_index][0].split('_')[0]
                                if previous_sym != '^' and relation_sym != 'frac' and relation_sym != 'sqrt':
                                    print('sup relation is wrong', lg_file)
                                    sys.exit()
                            if sym_relation[sym][min_index][1] == 'Below':
                                previous_sym = id_caption[id_caption.index(sym) - 1]
                                relation_sym = sym_relation[sym][min_index][0].split('_')[0]
                                if previous_sym != '_' and relation_sym != 'frac':
                                    print('sub relation is wrong', lg_file)
                                    sys.exit()
                    f_out.write(out_str + '\n')
                out_str = '</s>\t-1\t' + sym_segment[id_caption[-1]] + '\tEnd' + '\n'
                f_out.write(out_str)
            else:
                out_str = '</s>\t-1\t' + sym_segment[id_caption[-1]] + '\tEnd' + '\n'
                f_out.write(out_str)
            f_out.close()

            process_num += 1
            if process_num / 1000 == process_num * 1.0 / 1000:
                print('process files', process_num)

def gen_feature_pkl_v3(root_path, paths, feature_path, mask_path, out_file, out_mask_file):
    f_out = open(out_file, 'w')
    f_out_mask = open(out_mask_file, 'w')
    process_num = 0
    features = {}
    masks = {}
    for path in paths:
        file_list  = os.listdir(root_path + path)
        for file_name in file_list:
            # key = file_name[:-4] # remove suffix .srd
            key = file_name[:-3] # remove suffix .lg
            feature_file = feature_path + key + '.ascii'
            mat = numpy.loadtxt(feature_file)
            features[key] = mat
            mask_file = mask_path + key + '_mask.txt'
            mmat = numpy.loadtxt(mask_file)
            masks[key] = mmat
            process_num = process_num + 1
            if process_num / 500 == process_num * 1.0 / 500:
                print('process files', process_num)

    print('load ascii file done. files number ', process_num)

    pkl.dump(features, f_out)
    pkl.dump(masks, f_out_mask)
    print('save file done')
    f_out.close()
    f_out_mask.close()

def gen_srd_label_v5(root_path, paths, out_label_path, feature_file, mask_file, outpkl_label_file, out_file_align, out_file_related_align):

    alignment = {}
    related_alignment = {}
    label_lines = {}
    process_num = 0

    for path in paths:
        feature_file = feature_file.replace('test', path)
        mask_file = mask_file.replace('test', path)
        outpkl_label_file = outpkl_label_file.replace('test', path)
        out_file_align = out_file_align.replace('test', path)
        out_file_related_align = out_file_related_align.replace('test', path)

        feature_fp = open(feature_file)
        features = pkl.load(feature_fp)
        mask_fp = open(mask_file)
        masks = pkl.load(mask_fp)

        f_out_align = open(out_file_align, 'w')
        f_out_related_align = open(out_file_related_align, 'w')
        out_label_fp = open(outpkl_label_file, 'w')

        file_list  = os.listdir(root_path + path)
        for file_name in file_list:
            key = file_name[:-4] # remove suffix .srd
            if os.path.exists(out_label_path):
                out_label_file = out_label_path + '/' + key + '.label'
            else:
                os.mkdir(out_label_path)
                out_label_file = out_label_path + '/' + key + '.label'
            f_out_label = open(out_label_file, 'w')
            with open(root_path + path + '/' + file_name) as f:
                lines = f.readlines()
                wordNum = 0
                align_list = []
                realign_list = []
                label_strs = []
                for line in lines:
                    parts = line.strip().split('\t')
                    if len(parts) == 5:
                        wordNum += 1
                        sym = parts[0]
                        align_list.append(parts[1])
                        related_sym = parts[2]
                        realign_list.append(parts[3])
                        relation = parts[4]
                        string = sym + '\t' + related_sym + '\t' + relation
                        label_strs.append(string)
                        f_out_label.write(string + '\n')
                    else:
                        print('illegal line', key)
                        sys.exit()
                f_out_label.close()
                label_lines[key] = label_strs

                fea = features[key]
                mask = masks[key]
                align = numpy.zeros([fea.shape[0], wordNum], dtype='int8')
                realign = numpy.zeros([fea.shape[0], wordNum], dtype='int8')
                penup_index = numpy.where(fea[:,-1] == 1)[0] # 0 denote pen down, 1 denote pen up
                pp_start = 0
                for pi in range(len(penup_index)-1):
                    half_pad_num = 3
                    penup_index[pi] += half_pad_num
                    pp_start = penup_index[pi] + 1
                # if len(mask) != mask.sum():
                #     print key
                #     print penup_index
                #     sys.exit()

                wordNum = -1
                for align_str in align_list:
                    wordNum += 1
                    align_str_parts = align_str.split(', ')
                    for i in range(len(align_str_parts)):
                        pos = int(align_str_parts[i])
                        if pos == -1:
                            continue
                        elif pos == 0:
                            align[0:(penup_index[pos]+1), wordNum] = 1
                        else:
                            align[(penup_index[pos-1]+1):(penup_index[pos]+1), wordNum] = 1

                wordNum = -1
                for realign_str in realign_list:
                    wordNum += 1
                    realign_str_parts = realign_str.split(', ')
                    for i in range(len(realign_str_parts)):
                        pos = int(realign_str_parts[i])
                        if pos == -1:
                            continue
                        elif pos == 0:
                            realign[0:(penup_index[pos]+1), wordNum] = 1
                        else:
                            realign[(penup_index[pos-1]+1):(penup_index[pos]+1), wordNum] = 1

                alignment[key] = align
                related_alignment[key] = realign

            process_num = process_num + 1
            if process_num / 500 == process_num * 1.0 / 500:
                print('process files', process_num)

            pkl.dump(alignment, f_out_align)
            pkl.dump(related_alignment, f_out_related_align)
            pkl.dump(label_lines, out_label_fp)
            print(f'save {path} file done')
            f_out_align.close()
            f_out_related_align.close()
            out_label_fp.close()

    print('process files number ', process_num)


## Data Building

In [5]:
from glob import glob
import xml.etree.ElementTree as ET

In [6]:
def load_dict(dictFile):
    fp=open(dictFile)
    stuff=fp.readlines()
    fp.close()
    lexicon={}
    for l in stuff:
        w=l.strip().split()
        lexicon[w[0]]=int(w[1])

    return lexicon

def inkml_to_latex(root_path, paths, ns={"inkml": "http://www.w3.org/2003/InkML"}):
    # LaTeX caption file
    f = open(root_path + 'train_data_v1.txt', 'w')
    for path in paths:
        inkml_dir = root_path + path
        for inkml_file in glob(inkml_dir + "/*.inkml"):
            try:
                # Parse InkML file
                tree = ET.parse(inkml_file)
                root = tree.getroot()

                # Extract LaTeX string
                latex_string = root.find('.//inkml:annotation[@type="truth"]', ns)

                # Cache LaTeX string in a LaTeX caption file
                f.write(f'{inkml_file}\t{latex_string.text.strip(" $")}\n')
            except ET.ParseError as e:
                print(f'ET ParseError at {inkml_file}: {e}')
    f.close()

def default_folder(path):
   if os.path.exists(path):
      pass
   else:
      os.makedirs(path)


inkml_path = "/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/"
out_path = inkml_path
# out_path = './CACHED_CROHME/'

paths = ['train', 'val', 'test']
inkml_out = 'inkml_norm_id_r1/'
lg_out = 'LG_norm_id_r1/'
lg2_out = 'LG_norm_v2_r1/'
srd_out = 'srd_r1/'

default_folder(out_path + inkml_out)
default_folder(out_path + lg_out)
default_folder(out_path + lg2_out)
default_folder(out_path + srd_out)

feature_pkl = '16-9feature-test-dis-0.005-revise-pad-v5.pkl'
mask_pkl = '16-9feature-test-dis-0.005-revise-pad-v5-mask.pkl'
label_pkl = 'test-label-r1.pkl'
align_pkl = 'align-test-dis-0.005-revise-pad-v5-r1.pkl'
realign_pkl = 'related-align-test-dis-0.005-revise-pad-v5-r1.pkl'
dpkl = [feature_pkl, mask_pkl, label_pkl, align_pkl, realign_pkl]
datasets = [out_path + 'PKL/' + d for d in dpkl]

inkml_to_latex(out_path, paths)

crohmelib_bin_path = '/content/drive/MyDrive/Colab Notebooks/' #'bin/'
latex_caption_file = out_path + 'train_data_v1.txt'
out_label_path = datasets[2][:-4] + '/'
feature_path = datasets[0][:-4] + '/'
mask_path = datasets[1][:-4] + '/'

# hmer.norm_id_inkml(inkml_path, out_path + inkml_out, paths) # normalize inkml
inkml2lg(inkml_path, out_path + lg_out, paths, crohmelib_bin_path) # inkml -> lg
norm_lg_v2(out_path + lg_out, out_path + lg2_out, paths) # normalize lg
# latex caption of all inkml files input
get_srd_label(out_path + lg2_out, out_path + srd_out, paths, latex_caption_file) # lg -> srd

gen_feature_pkl_v3(out_path + lg2_out, paths, feature_path, mask_path, dpkl[0], dpkl[1]) # lg -> pickle feature, mask
gen_srd_label_v5(out_path + srd_out, paths, out_label_path, datasets[0], datasets[1], datasets[2], datasets[3], datasets[4]) # srd -> pickle label, align, realign

def srd_to_seq_label(srd_file, inkml_path_prefix=""):
    """
    Converts an SRD file into a flat symbol-relation sequence label.

    Returns: A string like:
    crohme2019/train/formulaire002-equation068.inkml    x Sup 2 NoRel + Right 2 Right x Right \sqrt Inside 2 NoRel + Right 1
    """
    symbol_chain = []
    last_symbol = None
    with open(srd_file, 'r') as f:
        lines = [line.strip().split('\t') for line in f if line.strip()]

    for line in lines:
        if line[0] == '<s>' or line[0] == '</s>':
            continue  # Skip start/end tags

        current_symbol = line[1]
        relation_type = line[3] if len(line) == 4 else "NoRel"

        if last_symbol is None:
            symbol_chain.append(current_symbol)
        else:
            symbol_chain.extend([relation_type, current_symbol])
        last_symbol = current_symbol

    # Build output: relative inkml path + tab + sequence
    base_name = os.path.basename(srd_file).replace('.srd', '.inkml')
    dir_name = os.path.basename(os.path.dirname(srd_file))
    relative_inkml_path = os.path.join(dir_name, base_name)

    return f"{relative_inkml_path}\t{' '.join(symbol_chain)}"

def convert_all_srd_in_dir(srd_path, paths, out_path, output_txt):
    for path in paths:
        with open(out_path + path + '_' + output_txt, 'w') as out_f:
            srd_dir = srd_path + '/' + path
            for root, _, files in os.walk(srd_dir):
                for fname in files:
                    if fname.endswith(".srd"):
                        srd_path = os.path.join(root, fname)
                        label_line = srd_to_seq_label(srd_path)
                        out_f.write(label_line + '\n')

convert_all_srd_in_dir(out_path + srd_out, paths, out_path, 'annotation.txt')

print(os.listdir(out_path))
print(os.system(f'cat {out_path}/train_annotation.txt'))


Streaming output truncated to the last 5000 lines.
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/LG_norm_id_r1/test/form_5_312_E1557.lg
process files 2234
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/test/CROHME2023_testform_5_223_E1114.inkml
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/LG_norm_id_r1/test/form_5_223_E1114.lg
process files 2235
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/test/CROHME2023_testform_310_E2478.inkml
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/LG_norm_id_r1/test/form_310_E2478.lg
process files 2236
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/test/CROHME2023_testform_305_E2439.inkml
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/LG_norm_id_r1/test/form_305_E2439.lg
process files 2237
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/test/CROHME2023_testform_334_E2669.inkml
/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/LG_norm_id_r1/test/form_

IsADirectoryError: [Errno 21] Is a directory: '/content/drive/MyDrive/Colab Notebooks/TC11_CROHME23/INKML/LG_norm_id_r1/train/CROHME2023_train'